# 12 — Evaluate behavior, not just prose
Objective checks should be plain Python. A model-judged score can be added
later, but approval, citation IDs, and bounded retries must be testable code.



**Basic lesson 12 of 13.**


## Problem

A polished answer can still bypass approval or contain invalid citations.


## Concept

Objective checks encode non-negotiable behavior before subjective scoring.

An evaluation case is a named input and a measurable check. Separate objective safety contracts from subjective answer quality, because prose can look good while a boundary is broken.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly   ◀ YOU ARE HERE
```

Every lesson adds one row to this system. The rows below the marker are still the same code from earlier lessons.


## Architecture of this lesson

```text
known cases ──▶ deterministic harness checks
                   ├──▶ ROI contract
                   ├──▶ report/source contract
                   └──▶ approval boundary
                              │
                         PASS / FAIL
```

test cases → deterministic checks → pass/fail evidence. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Define independent cases and their plain-Python pass/fail conditions

Define independent cases and their plain-Python pass/fail conditions.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from workshop.control import mock_email, validate_report
from workshop.schemas import DecisionReport, Evidence
from workshop.tools import calculate_annual_roi

cases = [
    ("roi scenario", lambda: calculate_annual_roi(45_000, 30_000) == 50.0),
    ("zero cost rejected", lambda: rejects_zero_cost()),
    ("email blocked", lambda: mock_email(example_report, approved=False, action_id="eval-denied").startswith("BLOCKED")),
    ("citations valid", lambda: validate_report(example_report)[0]),
]


def rejects_zero_cost() -> bool:
    try:
        calculate_annual_roi(45_000, 0)
    except ValueError:
        return True
    return False


example_report = DecisionReport(
    recommendation="pilot",
    rationale="The possible benefit merits a pilot, while security gaps block full adoption.",
    evidence=[
        Evidence(source_id="pricing.md", claim="Annual benefit is estimated at $45,000."),
        Evidence(source_id="security.md", claim="Deletion guarantees are unverified."),
    ],
)



## Run the checks
Extend the set with a model run after establishing this deterministic base.



## Step 2 — Run every check and use a failing case to identify the broken boundary

Run every check and use a failing case to identify the broken boundary.


In [ ]:
for name, check in cases:
    print(("PASS" if check() else "FAIL"), name)



## Bonus scenario — a self-evaluation loop
Plain-Python checks can grade any output (inspired by CrewAI's *Self Evaluation Loop* flow).
Generate candidates, score each one, keep the first that passes.



In [ ]:
def tagline_checks(tagline: str, product: str) -> dict[str, bool]:
    return {
        "mentions product": product.lower() in tagline.lower(),
        "10 words or fewer": len(tagline.split()) <= 10,
        "no exclamation mark": "!" not in tagline,
    }


candidates = [
    "TrailMix Bars are the absolute best snack for every single hiker out there!",
    "TrailMix Bars: fuel for the next ridge.",
]
for tagline in candidates:
    checks = tagline_checks(tagline, "TrailMix Bars")
    print(("PASS" if all(checks.values()) else "FAIL"), tagline, [name for name, ok in checks.items() if not ok])


## Failure mode to know about

A prose-quality score cannot catch a bypassed approval gate. Keep deterministic safety checks as release blockers and use model-judged scores only as an additional signal.


## Production implication

Run safety and contract cases in CI. Keep a small labeled set of grounded-answer examples for model or prompt changes.


## Your turn

Add a case for a report containing an unknown source ID.

**Checkpoint:** A failing case identifies a specific harness boundary to repair.
